# SmartBank AI: Bank Term Deposit Subscription Prediction
### Production-Grade Machine Learning System with Zero Data-Leakage Architecture
**Gupio Campus Placement Drive - AI/ML Practical Assignment**

---

## 1. Executive Summary & Problem Formulation
Direct marketing campaigns (telemarketing) for retail banking products like **term deposits** are resource-intensive. Reaching out indiscriminately to all clients creates unnecessary operational cost and customer annoyance.

### Core Task
Predict whether a retail banking client will subscribe to a term deposit (`y = 'yes'` / `'no'`), using **strictly information available immediately before the outbound contact begins**.

### Key Technical Requirements Addressed:
1. **Strict Leakage Prevention:** Elimination of post-contact features (`duration`, `contact`, `day`, `month`, `campaign`).
2. **Class Imbalance Awareness:** 88.3% non-subscribers vs 11.7% subscribers. Evaluation prioritizes ROC-AUC, PR-AUC, Precision, Recall, and F1 over misleading accuracy.
3. **Modular Preprocessing Pipeline:** Scikit-Learn `ColumnTransformer` + `Pipeline` fitted strictly on training data.
4. **Model Comparison:** Baseline Logistic Regression vs Nonlinear Random Forest vs HistGradientBoosting.
5. **Causal vs Predictive Distinction:** Careful interpretation of feature importance.


In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries imported successfully!')

## 2. Dataset Loading & Initial Inspection
We load the official UCI Bank Marketing dataset (`data/bank-full.csv`), which contains 45,211 rows and 17 columns separated by semicolons (`;`).

In [2]:
data_path = '../data/bank-full.csv'
if not os.path.exists(data_path):
    data_path = 'data/bank-full.csv'

df = pd.read_csv(data_path, sep=';')

print('Dataset Shape:', df.shape)
print('Number of Rows:', df.shape[0])
print('Number of Columns:', df.shape[1])
display(df.head())

## 3. Data Types, Missing Values, and Duplicates Inspection

In [3]:
print('=== COLUMN DATA TYPES ===')
print(df.dtypes)

print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

print('\n=== DUPLICATE RECORDS ===')
print('Duplicate rows count:', df.duplicated().sum())

## 4. Target Distribution & Class Imbalance Analysis

In [4]:
print('Target Counts:')
print(df['y'].value_counts())

print('\nTarget Proportions (%):')
print(df['y'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='y', palette=['#3498db', '#e74c3c'])
plt.title('Term Deposit Subscription Distribution (Imbalanced: 88.3% vs 11.7%)', fontweight='bold')
plt.xlabel('Subscribed (y)')
plt.ylabel('Number of Clients')
plt.tight_layout()
plt.show()

## 5. Leakage Prevention Protocol & Feature Selection

### Excluded Features (Temporal & Operational Leakage):
- `duration`: Call length is only known after the phone call occurs. Using it creates severe target leakage.
- `contact`, `day`, `month`, `campaign`: Pertain to the current campaign execution logistics.

### Permitted Predictors (11 Features):
- **Numerical:** `age`, `balance`, `pdays`, `previous`
- **Categorical / Binary:** `job`, `marital`, `education`, `default`, `housing`, `loan`, `poutcome`

In [5]:
excluded_features = ['contact', 'day', 'month', 'duration', 'campaign']
X = df.drop(columns=['y'] + excluded_features)
y = df['y'].map({'no': 0, 'yes': 1}).astype(int)

print('Features used in model:', X.columns.tolist())
print('Number of predictors:', X.shape[1])
print('Positive class count (Subscribed):', y.sum())

## 6. Reproducible Stratified Train-Test Split (80/20)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train shape: {X_train.shape} (Positives: {y_train.sum()}, {y_train.mean()*100:.2f}%)')
print(f'Test shape:  {X_test.shape} (Positives: {y_test.sum()}, {y_test.mean()*100:.2f}%)')

## 7. Leakage-Safe Scikit-Learn Preprocessing Pipeline

In [7]:
numerical_cols = ['age', 'balance', 'pdays', 'previous']
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'poutcome']

num_pipe = Pipeline(steps=[('scaler', StandardScaler())])
cat_pipe = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, numerical_cols),
        ('cat', cat_pipe, categorical_cols)
    ],
    remainder='drop'
)
print('Preprocessor built successfully!')

## 8. Model Training & 5-Fold Stratified Cross-Validation

In [8]:
models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_split=10,
            class_weight='balanced', random_state=42, n_jobs=-1
        ))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=['roc_auc', 'f1', 'recall', 'precision'])
    cv_results[name] = scores
    print(f'=== {name} 5-Fold CV Scores ===')
    print(f'  ROC-AUC:   {scores["test_roc_auc"].mean():.4f} +/- {scores["test_roc_auc"].std():.4f}')
    print(f'  F1-Score:  {scores["test_f1"].mean():.4f} +/- {scores["test_f1"].std():.4f}')
    print(f'  Recall:    {scores["test_recall"].mean():.4f} +/- {scores["test_recall"].std():.4f}')
    print(f'  Precision: {scores["test_precision"].mean():.4f} +/- {scores["test_precision"].std():.4f}\n')

## 9. Final Holdout Test Set Evaluation (n = 9,043)

In [9]:
test_metrics = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    
    test_metrics[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
        'Macro F1': macro_f1, 'ROC-AUC': roc_auc, 'PR-AUC': pr_auc, 'CM': cm, 'Prob': y_prob
    }
    
    print(f'=== {name} Unseen Test Report ===')
    print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))
    print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}\n')

## 10. Evaluation Charts: Confusion Matrices and ROC Curves

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (name, metrics) in zip(axes, test_metrics.items()):
    sns.heatmap(metrics['CM'], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred: No', 'Pred: Yes'], yticklabels=['Actual: No', 'Actual: Yes'])
    ax.set_title(f'{name}\nROC-AUC = {metrics["ROC-AUC"]:.4f}', fontweight='bold')

plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(7, 5))
for name, metrics in test_metrics.items():
    fpr, tpr, _ = roc_curve(y_test, metrics['Prob'])
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {metrics["ROC-AUC"]:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Baseline (0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves on Unseen Test Set', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 11. Feature Importance & Model Interpretation

In [11]:
rf = models['Random Forest']
cat_encoder = rf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()
feature_names = numerical_cols + cat_names

importances = rf.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feat_df.head(12), x='Importance', y='Feature', palette='viridis')
plt.title('Top 12 Features Driving Deposit Propensity (Gini Importance)', fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Model Inference Demonstration on Sample Profiles

In [12]:
sample_prospects = pd.DataFrame([
    {
        'age': 68, 'job': 'retired', 'marital': 'married', 'education': 'tertiary',
        'default': 'no', 'balance': 18500, 'housing': 'no', 'loan': 'no',
        'pdays': 180, 'previous': 3, 'poutcome': 'success'
    },
    {
        'age': 28, 'job': 'blue-collar', 'marital': 'married', 'education': 'secondary',
        'default': 'no', 'balance': 200, 'housing': 'yes', 'loan': 'yes',
        'pdays': -1, 'previous': 0, 'poutcome': 'unknown'
    }
])

rf_model = models['Random Forest']
probs = rf_model.predict_proba(sample_prospects)[:, 1]
preds = rf_model.predict(sample_prospects)

for i, (prob, pred) in enumerate(zip(probs, preds)):
    print(f'Customer Profile {i+1}:')
    print(f'  Subscription Probability: {prob*100:.2f}%')
    print(f'  Classification Decision:  {"Yes (Subscribe)" if pred == 1 else "No"}\n')